In [25]:
# Python
# Connected Device Operations Agent
# Version 1 - FleetResolve AI Agent




In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# ============================================================
# 1. SYNTHETIC OPERATIONAL DATA
# ============================================================
# Device, user, and incident data used by our agent

In [28]:
# Add 5 Devices

devices = {
    "AX-1001": {
        "device_id": "AX-1001",
        "device_type": "Body Camera",
        "status": "Online",
        "battery": 87,
        "firmware": "5.2.1",
        "last_seen": "2026-08-16 14:20",
        "assigned_user": "USR-001"
    },
    "AX-1002": {
        "device_id": "AX-1002",
        "device_type": "Body Camera",
        "status": "Offline",
        "battery": 8,
        "firmware": "5.1.9",
        "last_seen": "2026-08-16 10:12",
        "assigned_user": "USR-002"
    },
    "AX-1003": {
        "device_id": "AX-1003",
        "device_type": "In-Car Camera",
        "status": "Online",
        "battery": 62,
        "firmware": "5.2.1",
        "last_seen": "2026-08-16 14:25",
        "assigned_user": "USR-003"
    },
    "AX-1004": {
        "device_id": "AX-1004",
        "device_type": "Body Camera",
        "status": "Offline",
        "battery": 76,
        "firmware": "5.0.4",
        "last_seen": "2026-08-16 09:45",
        "assigned_user": "USR-004"
    },
    "AX-1005": {
        "device_id": "AX-1005",
        "device_type": "Sensor",
        "status": "Online",
        "battery": 94,
        "firmware": "3.4.0",
        "last_seen": "2026-08-16 14:28",
        "assigned_user": "USR-005"
    }
}

In [29]:
# Users of Device
users = {
    "USR-001": {
        "user_id": "USR-001",
        "name": "Alex Morgan",
        "department": "Patrol",
        "location": "Austin"
    },
    "USR-002": {
        "user_id": "USR-002",
        "name": "Jordan Lee",
        "department": "Patrol",
        "location": "Austin"
    },
    "USR-003": {
        "user_id": "USR-003",
        "name": "Taylor Smith",
        "department": "Traffic",
        "location": "Dallas"
    },
    "USR-004": {
        "user_id": "USR-004",
        "name": "Morgan Chen",
        "department": "Patrol",
        "location": "Dallas"
    },
     "USR-005": {
        "user_id": "USR-005",
        "name": "Casey Brown",
        "department": "Operations",
        "location": "Houston"
    }
}

In [30]:
# Add Incidents
incidents = {
    "AX-1001": [],

    "AX-1002": [
        {
            "incident_id": "INC-201",
            "type": "Battery Failure",
            "date": "2026-07-20",
            "resolution": "Battery replaced"
        },
        {
            "incident_id": "INC-245",
            "type": "Connectivity Loss",
            "date": "2026-08-02",
            "resolution": "Device restarted"
        }
    ],
    "AX-1003": [],

    "AX-1004": [
        {
            "incident_id": "INC-198",
            "type": "Firmware Issue",
            "date": "2026-07-15",
            "resolution": "Firmware updated"
        }
    ],

    "AX-1005": []
}

In [31]:
# ============================================================
# 2. AGENT TOOLS
# ============================================================
# Tools exposed to the LLM
# Each tool has a defined name, description, input, and output

In [32]:
from langchain.tools import tool

In [33]:
# Turn that into tools that Ai Agnets can use
@tool
def get_device(device_id: str) -> dict:
    """Get the current status and metadata for a connected device."""
    return devices.get(
        device_id,
        {"error": f"Device {device_id} not found"}
    )


@tool
def get_user(user_id: str) -> dict:
    """Get information about the user assigned to a device."""
    return users.get(
        user_id,
        {"error": f"User {user_id} not found"}
    )


@tool
def get_incident_history(device_id: str) -> list:
    """Get historical incidents associated with a connected device."""
    return incidents.get(device_id, [])



In [34]:
# ============================================================
# 3. MANUAL TOOL VALIDATION
# ============================================================
# Verify that each underlying capability works independently

In [35]:
# Test Manually
print("DEVICE:")
print(get_device.invoke({"device_id": "AX-1002"}))

print("\nUSER:")
print(get_user.invoke({"user_id": "USR-002"}))

print("\nINCIDENT HISTORY:")
print(get_incident_history.invoke({"device_id": "AX-1002"}))


DEVICE:
{'device_id': 'AX-1002', 'device_type': 'Body Camera', 'status': 'Offline', 'battery': 8, 'firmware': '5.1.9', 'last_seen': '2026-08-16 10:12', 'assigned_user': 'USR-002'}

USER:
{'user_id': 'USR-002', 'name': 'Jordan Lee', 'department': 'Patrol', 'location': 'Austin'}

INCIDENT HISTORY:
[{'incident_id': 'INC-201', 'type': 'Battery Failure', 'date': '2026-07-20', 'resolution': 'Battery replaced'}, {'incident_id': 'INC-245', 'type': 'Connectivity Loss', 'date': '2026-08-02', 'resolution': 'Device restarted'}]


In [36]:
# ============================================================
# 4. ENVIRONMENT & CONFIGURATION
# ============================================================
# API keys, imports, model configuration
# This is section is about Authentication
# OS lets Python to interact with environment variables
# lets colab retrieve a secret you have stored in Colab Secrets
# Takes the Groq API key and make it avilable to the Python Process

In [37]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


In [38]:
!pip install -q groq

In [39]:
# The API key makes the LLM available to python process.
# Because LLM isn't running inside notebook
# The request is authorized to use the API
# The API key gives python notebook access to groq's model service
# Groq client + list of models

from groq import Groq
print("API key loaded:", bool(os.environ["GROQ_API_KEY"]))

client = Groq(api_key=os.environ["GROQ_API_KEY"])

models = client.models.list()

for model in models.data:
    print(model.id)

API key loaded: True
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
groq/compound
allam-2-7b
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
whisper-large-v3
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
groq/compound-mini
openai/gpt-oss-120b


In [40]:
# This is environment setup
# Install the Python packcage into my colab environment

!pip install -q -U langchain-groq

In [41]:
# Import LangChain integration for Groq
# Create LLM interface that LangChain can work with
# LangChain is the agent/applicatin framework we're using
# We are providing the connected-device capabilities
# We are teaching the system about body camera, device-id, incidents by creating the tools and their decriptions

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("LLM ready!")

LLM ready!


In [42]:
# Test that LLM is added
response = llm.invoke(
    "In one sentence, explain what an AI agent is."
)

print(response.content)

An AI agent is a software entity that perceives its environment through data inputs, processes that information using algorithms or models, and takes autonomous actions to achieve specific goals.


In [43]:
# ============================================================
# 5. LLM + TOOL BINDING
# ============================================================
# Connect the LLM to the available tools

In [44]:
# Bind the three tools created above to the LLM

tools = [
    get_device,
    get_user,
    get_incident_history
]

llm_with_tools = llm.bind_tools(tools)

print("Tools available to the LLM:")

for t in tools:
    print("-", t.name)

Tools available to the LLM:
- get_device
- get_user
- get_incident_history


In [45]:
# ============================================================
# 7. AGENT LOOP — OBSERVE → DECIDE → ACT
# ============================================================
# Feed tool results back to the LLM and allow it to determine
# whether another tool is needed

# The deterministic workflow
# 1. get_device
# 2. get_user
# 3. get_incidents

# The agent should instead be able to do
# 1. Decide what to investigate
# 2. Execute one action
# 3. Observe result
# 4. Decide next action
# 5. Execute
# 6. Observe
# 7. Decide whether finished

# That's the loop



In [46]:
# Above demonstrates that first I manually demonstrated an agent loop
# Now I built an agent loop
# The loop is
#             USER
#                │
#                ▼
#               LLM
#                │
#          Does it need a tool?
#           /              \
#         YES               NO
#          │                 │
#          ▼                 ▼
#       TOOL             FINAL ANSWER
#          │
#          ▼
#       RESULT
#          │
#          └──────────► LLM
#                         │
#                    Decide again

In [47]:
# ============================================================
# 9. OPERATIONAL KNOWLEDGE
# ============================================================
# Troubleshooting rules and service policies used by the agent
# The agent can retrieve operational facts through tools
# Operational knowledge provides the rules needed to interpret those facts and make a recommendation
# The goal is to seperate:
# What the system knows - tool results
# What the organization knows - operational knowledge
# What the agent recommends - reasoing based on both
# lets create troubleshooting knowledge base

troubleshooting_knowledge = """
Connected Device Troubleshooting Guidelines:

1. A device with battery below 10% should be considered a low-battery condition.

2. A device that is offline with battery below 10% should receive an
   immediate service review recommendation.

3. A previous battery failure indicates that battery health should be
   considered when evaluating a current low-battery condition.

4. A previous connectivity loss indicates that connectivity should be
   considered when evaluating a current offline condition.

5. Recommendations should be based on available evidence. Do not claim
   that a component has failed unless the available evidence supports it.
"""

print(troubleshooting_knowledge)


Connected Device Troubleshooting Guidelines:

1. A device with battery below 10% should be considered a low-battery condition.

2. A device that is offline with battery below 10% should receive an
   immediate service review recommendation.

3. A previous battery failure indicates that battery health should be
   considered when evaluating a current low-battery condition.

4. A previous connectivity loss indicates that connectivity should be
   considered when evaluating a current offline condition.

5. Recommendations should be based on available evidence. Do not claim
   that a component has failed unless the available evidence supports it.



In [48]:
# The agent can now reason
# 8% battery + offline + previous battery failure + previous connectivity loss + Immediate service review
# There is still no ability to take action

## What we are building today
# GENERAL AGENT V1
# Can investigate
#        ↓
# OPERATIONAL KNOWLEDGE
# Knows how to interpret conditions
#        ↓
# RECOMMENDATION
# Can recommend
#        ↓
# GUARDRAILS
# Knows what it must NOT do
#         ↓
# HUMAN APPROVAL
# Human controls consequential action

In [49]:
# ============================================================
# Make operational knowledge available to the agent
# ============================================================

agent_question = f"""
Investigate device AX-1002.

Tell me:
1. Who is the device assigned to?
2. Whether it has experienced similar problems.
3. Based on the troubleshooting guidelines, what should be recommended?

Use the following operational knowledge when making your recommendation:

{troubleshooting_knowledge}
"""

In [50]:
from langchain_core.messages import HumanMessage, ToolMessage

In [51]:
question_2 = agent_question
messages = [
    HumanMessage(content=question_2)
]
print("QUESTION:")
print(question_2)

print("\nMESSAGE CREATED:")
print(messages)

QUESTION:

Investigate device AX-1002.

Tell me:
1. Who is the device assigned to?
2. Whether it has experienced similar problems.
3. Based on the troubleshooting guidelines, what should be recommended?

Use the following operational knowledge when making your recommendation:


Connected Device Troubleshooting Guidelines:

1. A device with battery below 10% should be considered a low-battery condition.

2. A device that is offline with battery below 10% should receive an
   immediate service review recommendation.

3. A previous battery failure indicates that battery health should be
   considered when evaluating a current low-battery condition.

4. A previous connectivity loss indicates that connectivity should be
   considered when evaluating a current offline condition.

5. Recommendations should be based on available evidence. Do not claim
   that a component has failed unless the available evidence supports it.



MESSAGE CREATED:
[HumanMessage(content='\nInvestigate device AX-100

In [52]:
def run_agent (messages, max_steps = 10):

  for step in range(max_steps):

      print(f"\n--- STEP {step + 1} ---")

      response = llm_with_tools.invoke(messages)

      messages.append(response)

      # If the LLM does not request another tool,
      # it has decided it has enough information.
      if not response.tool_calls:
          print("AGENT FINISHED")
          print("\nFINAL ANSWER:")
          print(response.content)
          return response

      # Execute each tool requested by the LLM
      for tool_call in response.tool_calls:

          tool_name = tool_call["name"]
          tool_args = tool_call["args"]

          print(f"TOOL: {tool_name}")
          print(f"ARGS: {tool_args}")

          selected_tool = tool_map[tool_name]

          result = selected_tool.invoke(tool_args)

          print(f"RESULT: {result}")

          messages.append(
              ToolMessage(
                  content=str(result),
                  tool_call_id=tool_call["id"]
              )
          )

      print("Agent stopped after reaching the maximum number of steps.")

In [53]:
## Day 2 Observation - Operational Knowledge
# Adding explicit troubleshooting guidance changed the agent's behavior.
# The agent now grounded its recommendation in the provided operational guidelines.
# However, it also inferred specific actions that were not explicitly authorized by those guidelines, such as battery replacement, antenna checks, firmware updates, and functional testing.
# This raises the next engineering question:
# **How do we constrain an agent's recommendations to what it is actually authorized to recommend or execute?**
# This is the transition from operational knowledge to decision policy and guardrails.


In [54]:
# ============================================================
# 10. RECOMMENDATION & DECISION POLICY
# ============================================================
# Agent evaluates device conditions and recommends an action
# Operational knowledge helps the agent interpret facts
# Decision policy defines what the agent is authorized to recommend based on those facts
# The agent should not infer operational authority from egenrak troubleshooting knowledge

decision_policy = """
Decision Policy for Connected Devices:

1. If a device has battery below 10%, recommend a service review.

2. If a device is offline AND battery is below 10%, recommend an immediate
   service review.

3. If there is a previous battery failure, recommend checking battery health
   and replacement history.

4. If there is a previous connectivity loss, recommend verifying connectivity.

5. The agent may recommend investigation or service review.

6. The agent must NOT directly authorize component replacement, device
   deactivation, return-to-service, firmware changes, or other consequential
   operational actions.

7. If evidence is insufficient to make a recommendation, the agent must say
   that the evidence is insufficient and request human review.

8. The agent must not introduce specific diagnostic procedures, technical
   causes, or operational steps that are not supported by the available
   evidence or operational knowledge.

9. When the available evidence does not support a specific procedure, the
   agent should state the limitation rather than infer a procedure.
"""

print(decision_policy)


Decision Policy for Connected Devices:

1. If a device has battery below 10%, recommend a service review.

2. If a device is offline AND battery is below 10%, recommend an immediate
   service review.

3. If there is a previous battery failure, recommend checking battery health
   and replacement history.

4. If there is a previous connectivity loss, recommend verifying connectivity.

5. The agent may recommend investigation or service review.

6. The agent must NOT directly authorize component replacement, device
   deactivation, return-to-service, firmware changes, or other consequential
   operational actions.

7. If evidence is insufficient to make a recommendation, the agent must say
   that the evidence is insufficient and request human review.

8. The agent must not introduce specific diagnostic procedures, technical
   causes, or operational steps that are not supported by the available
   evidence or operational knowledge.

9. When the available evidence does not support a sp

In [55]:

agent_question = f"""
Investigate device AX-1002.

Tell me:
1. Who is the device assigned to?
2. Whether it has experienced similar problems.
3. What operational recommendation should be made?

Use the following operational knowledge:

{troubleshooting_knowledge}

Use the following decision policy to determine what you are authorized
to recommend:

{decision_policy}
Do not recommend or authorize actions outside this policy.
"""

In [56]:
question_2 = agent_question

messages = [
    HumanMessage(content=question_2)
]

response = run_agent(messages)


--- STEP 1 ---
TOOL: get_device
ARGS: {'device_id': 'AX-1002'}
RESULT: {'device_id': 'AX-1002', 'device_type': 'Body Camera', 'status': 'Offline', 'battery': 8, 'firmware': '5.1.9', 'last_seen': '2026-08-16 10:12', 'assigned_user': 'USR-002'}
Agent stopped after reaching the maximum number of steps.

--- STEP 2 ---
TOOL: get_user
ARGS: {'user_id': 'USR-002'}
RESULT: {'user_id': 'USR-002', 'name': 'Jordan Lee', 'department': 'Patrol', 'location': 'Austin'}
Agent stopped after reaching the maximum number of steps.

--- STEP 3 ---
TOOL: get_incident_history
ARGS: {'device_id': 'AX-1002'}
RESULT: [{'incident_id': 'INC-201', 'type': 'Battery Failure', 'date': '2026-07-20', 'resolution': 'Battery replaced'}, {'incident_id': 'INC-245', 'type': 'Connectivity Loss', 'date': '2026-08-02', 'resolution': 'Device restarted'}]
Agent stopped after reaching the maximum number of steps.

--- STEP 4 ---
AGENT FINISHED

FINAL ANSWER:
**1. Assigned user**  
- **Name:** Jordan Lee  
- **Department:** Patr

In [57]:
# ============================================================
# 11. HUMAN-IN-THE-LOOP
# ============================================================
# Consequential actions require human approval
# Next we build the human escalation mechanism
#               AGENT
#               ↓
#          Investigate
#                ↓
#          Make recommendation
#                ↓
#        Decision Policy
#                ↓
#           Guardrail
#          ↙         ↘
#     Allowed       Consequential
#        ↓               ↓
#    Continue       HUMAN REVIEW
# The important distinction is:
# Decision policy = what the agent is allowed to recommend.
# Guardrail = what the agent is prevented from doing.
# Human escalation = what happens when the boundary is reached.

In [58]:
# ============================================================
# 12. GUARDRAILS & FAILURE HANDLING
# ============================================================
# Define what the agent can do, cannot do, and what happens
# when tools or information are unavailable
# A decision policy defines what the agent is authorized to recommend.
# A guardrail defines actions the agent must not take without additional approval.
# When a recommendation crosses that boundary, the workflow should escalate to a human rather than allowing the agent to proceed.
#
## The key insight is that the guardrail is deterministic Python code, not another LLM judgment.
## That is important for the kind of operational systems you're thinking about: the LLM can be probabilistic, but certain boundaries can remain deterministic.

guardrails = {
    "requires_human_approval": [
        "component_replacement",
        "device_deactivation",
        "firmware_change",
        "return_to_service"
    ],
    "evidence_bounded_recommendation": True
}

print(guardrails)

def check_guardrail(action: str) -> dict:
    if action in guardrails["requires_human_approval"]:
        return {
            "status": "HUMAN_APPROVAL_REQUIRED",
            "action": action,
            "reason": "This action is consequential and requires human approval."
        }

    return {
        "status": "ACTION_ALLOWED",
        "action": action,
        "reason": "This action is within the agent's authorized scope."
    }

print(check_guardrail("component_replacement"))
print(check_guardrail("service_review"))

{'requires_human_approval': ['component_replacement', 'device_deactivation', 'firmware_change', 'return_to_service'], 'evidence_bounded_recommendation': True}
{'status': 'HUMAN_APPROVAL_REQUIRED', 'action': 'component_replacement', 'reason': 'This action is consequential and requires human approval.'}
{'status': 'ACTION_ALLOWED', 'action': 'service_review', 'reason': "This action is within the agent's authorized scope."}


In [59]:
## Escalation function
# Connect the guardrail to escalation

def escalate_to_human(action: str, reason: str) -> dict:
    return {
        "status": "ESCALATED_TO_HUMAN",
        "action": action,
        "reason": reason,
        "next_step": "Human review and approval required."
    }
def evaluate_action(action: str) -> dict:
    guardrail_result = check_guardrail(action)

    if guardrail_result["status"] == "HUMAN_APPROVAL_REQUIRED":
        return escalate_to_human(
            action,
            guardrail_result["reason"]
        )

    return guardrail_result

print("SERVICE REVIEW:")
print(evaluate_action("service_review"))

print("\nCOMPONENT REPLACEMENT:")
print(evaluate_action("component_replacement"))

SERVICE REVIEW:
{'status': 'ACTION_ALLOWED', 'action': 'service_review', 'reason': "This action is within the agent's authorized scope."}

COMPONENT REPLACEMENT:
{'status': 'ESCALATED_TO_HUMAN', 'action': 'component_replacement', 'reason': 'This action is consequential and requires human approval.', 'next_step': 'Human review and approval required.'}


In [60]:
## LLM recommendation
#        ↓
# Decision Policy
#         ↓
# Guardrail
#        ↓
# ┌───────────────┐
# │               │
# Allowed       Consequential
# │               │
# ↓               ↓
# Proceed       Human escalation

## The bigger learning
# The LLM doesn't need to be perfectly reliable for the system to be safe.
# The LLM proposes → deterministic policy constrains → deterministic guardrail enforces → human takes over when necessary.
# That's very close to the policy/automation thinking you already have from cellular infrastructure — except now the LLM can dynamically investigate and propose based on context.

In [61]:
##=========================================================================
# Evidence Validation
# What we have proven
# The agnet can invetsigate
# It can apply operayional knowledge
# It can follow decision policy
# Deterministic guardrails can block consequential actions
# FAIL - The LLM can still add plausible but unsupported procedures

## So teh architecture should be
# LLM investigates
#      ↓
# LLM proposes recommendation
#      ↓
# Evidence / Policy Validation
#      ↓
#   ┌───────┴───────┐
#   ↓               ↓
# PASS             FAIL
#   ↓               ↓
# Deliver        Revise / Human Review

## Evidence Validation
# Every operational recommendation must be traceable to either:
# 1. retrieved evidence, or
# 2. an explicit operational guideline / decision policy.

# The agent must not introduce unsupported diagnostic procedures,
# technical causes, or operational actions.

# If a recommendation cannot be supported, it should be flagged for human review

## The Recommendation should be clean and traceable example:
# Recommendation:
# Immediate service review

# Evidence:
# - Offline
# - Battery 8%
# - Previous battery failure
# - Previous connectivity loss

# Policy:
# - Offline + battery <10% → immediate service review

# Unsupported actions:
# None

In [62]:
# Evidence Validation
# Every operational recommendation must be supported by:

# 1. Retrieved evidence, or
# 2. Explicit operational knowledge / decision policy.

# The agent must not introduce unsupported diagnostic procedures,
# technical causes, or operational actions.

# If a recommendation cannot be supported, it should be flagged for human review.

supported_recommendations = [
    "immediate service review",
    "check battery health and replacement history",
    "verify connectivity"
]

print(supported_recommendations)

['immediate service review', 'check battery health and replacement history', 'verify connectivity']


In [63]:
# Create the Validator
def validate_recommendation(recommendation: str) -> dict:
    recommendation_lower = recommendation.lower()

    unsupported_items = []

    for item in [
        "battery replacement",
        "firmware change",
        "device deactivation",
        "return to service",
        "battery under load",
        "signal strength",
        "network settings",
        "charging practices"
    ]:
        if item in recommendation_lower:
            unsupported_items.append(item)

    if unsupported_items:
        return {
            "status": "HUMAN_REVIEW_REQUIRED",
            "reason": "Recommendation contains unsupported operational details.",
            "unsupported_items": unsupported_items
        }

    return {
        "status": "VALID",
        "reason": "Recommendation is within the defined evidence and policy boundary.",
        "unsupported_items": []
    }

In [64]:
# Validate the actula agent recommendation

agent_recommendation = response.content

validation_result = validate_recommendation(agent_recommendation)

print("AGENT RECOMMENDATION VALIDATION:")
print(validation_result)

AGENT RECOMMENDATION VALIDATION:
{'status': 'HUMAN_REVIEW_REQUIRED', 'reason': 'Recommendation contains unsupported operational details.', 'unsupported_items': ['firmware change']}


In [65]:
## Structured Recommendation Contract
# The agent must separate its output into four parts:

# 1. Recommendation — what the agent recommends
# 2. Evidence — facts supporting the recommendation
# 3. Policy Basis — the operational rule supporting the recommendation
# 4. Not Authorized — consequential actions that require human approval

# This structure allows downstream validation to distinguish a
# recommendation from actions that are explicitly not authorized

structured_evaluation_question = """
Investigate device AX-1002 and provide an operational recommendation.

Return your answer using exactly these sections:

RECOMMENDATION:
State only the action or recommendation supported by the evidence
and operational policy.

EVIDENCE:
List only facts retrieved from the available tools.

POLICY BASIS:
State the operational guideline that supports the recommendation.

NOT AUTHORIZED:
List consequential actions that require human approval.
Do not present these as recommendations.

Do not introduce technical causes, diagnostic procedures, or
operational steps that are not explicitly supported by the
available evidence or operational policy.
"""


In [66]:
structured_messages = [
    HumanMessage(content=structured_evaluation_question)
]

structured_response = run_agent(structured_messages)


--- STEP 1 ---
TOOL: get_device
ARGS: {'device_id': 'AX-1002'}
RESULT: {'device_id': 'AX-1002', 'device_type': 'Body Camera', 'status': 'Offline', 'battery': 8, 'firmware': '5.1.9', 'last_seen': '2026-08-16 10:12', 'assigned_user': 'USR-002'}
Agent stopped after reaching the maximum number of steps.

--- STEP 2 ---
TOOL: get_user
ARGS: {'user_id': 'USR-002'}
RESULT: {'user_id': 'USR-002', 'name': 'Jordan Lee', 'department': 'Patrol', 'location': 'Austin'}
Agent stopped after reaching the maximum number of steps.

--- STEP 3 ---
TOOL: get_incident_history
ARGS: {'device_id': 'AX-1002'}
RESULT: [{'incident_id': 'INC-201', 'type': 'Battery Failure', 'date': '2026-07-20', 'resolution': 'Battery replaced'}, {'incident_id': 'INC-245', 'type': 'Connectivity Loss', 'date': '2026-08-02', 'resolution': 'Device restarted'}]
Agent stopped after reaching the maximum number of steps.

--- STEP 4 ---
AGENT FINISHED

FINAL ANSWER:
**RECOMMENDATION:**  
Retrieve device AX‑1002 from the field for charg

In [67]:
structured_evaluation_question_v2 = f"""
Investigate device AX-1002 and provide an operational recommendation.

Use ONLY the following operational knowledge and decision policy.

OPERATIONAL KNOWLEDGE:
{troubleshooting_knowledge}

DECISION POLICY:
{decision_policy}

Return your answer using exactly these sections:

RECOMMENDATION:
State only the recommendation supported by the evidence and policy.

EVIDENCE:
List only facts retrieved from the available tools.

POLICY BASIS:
Identify the specific guideline or decision-policy rule supporting
the recommendation.

NOT AUTHORIZED:
List consequential actions that require human approval.
Do not present these as recommendations.

IMPORTANT:
- Do not invent policies, thresholds, procedures, technical causes,
  or operational actions.
- Do not use general domain knowledge that is not contained in the
  operational knowledge or decision policy above.
- If the evidence or policy is insufficient, explicitly say so
  and recommend human review.
"""

In [68]:
structured_messages_v2 = [
    HumanMessage(content=structured_evaluation_question_v2)
]

structured_response_v2 = run_agent(structured_messages_v2)


--- STEP 1 ---
TOOL: get_device
ARGS: {'device_id': 'AX-1002'}
RESULT: {'device_id': 'AX-1002', 'device_type': 'Body Camera', 'status': 'Offline', 'battery': 8, 'firmware': '5.1.9', 'last_seen': '2026-08-16 10:12', 'assigned_user': 'USR-002'}
Agent stopped after reaching the maximum number of steps.

--- STEP 2 ---
TOOL: get_incident_history
ARGS: {'device_id': 'AX-1002'}
RESULT: [{'incident_id': 'INC-201', 'type': 'Battery Failure', 'date': '2026-07-20', 'resolution': 'Battery replaced'}, {'incident_id': 'INC-245', 'type': 'Connectivity Loss', 'date': '2026-08-02', 'resolution': 'Device restarted'}]
Agent stopped after reaching the maximum number of steps.

--- STEP 3 ---
AGENT FINISHED

FINAL ANSWER:
**RECOMMENDATION:**  
Recommend an immediate service review, including checking battery health and verifying connectivity.

**EVIDENCE:**  
- Device status: Offline.  
- Battery level: 8% (below 10%).  
- Incident history includes a prior Battery Failure (2026‑07‑20).  
- Incident histo

In [69]:
## Add a new validator for the structured recommandation
# Can the validator distinguish an unauthorized action being recommended from an action merely being listed as NOT AUTHORIZED?
def validate_structured_recommendation(response_text: str) -> dict:
    """
    Validate only the RECOMMENDATION section.
    Actions listed under NOT AUTHORIZED should not trigger a failure.
    """

    text = response_text

    # Find the recommendation section
    start = text.find("RECOMMENDATION:")
    end = text.find("EVIDENCE:")

    if start == -1 or end == -1:
        return {
            "status": "HUMAN_REVIEW_REQUIRED",
            "reason": "Structured recommendation format is incomplete."
        }

    recommendation = text[start:end].lower()

    unsupported_items = []

    for item in [
        "battery replacement",
        "component replacement",
        "device deactivation",
        "firmware change",
        "return to service",
        "battery under load",
        "signal strength",
        "network settings",
        "charging practices"
    ]:
        if item in recommendation:
            unsupported_items.append(item)

    if unsupported_items:
        return {
            "status": "HUMAN_REVIEW_REQUIRED",
            "reason": "Recommendation contains unsupported operational details.",
            "unsupported_items": unsupported_items
        }

    return {
        "status": "VALID",
        "reason": "Recommendation is within the defined evidence and policy boundary.",
        "unsupported_items": []
    }

In [70]:
validation_v2 = validate_structured_recommendation(
    structured_response_v2.content
)

print("STRUCTURED RECOMMENDATION VALIDATION:")
print(validation_v2)


STRUCTURED RECOMMENDATION VALIDATION:
{'status': 'VALID', 'reason': 'Recommendation is within the defined evidence and policy boundary.', 'unsupported_items': []}


In [71]:
## Final Evidence Validation

# The structured recommendation was evaluated independently from the NOT AUTHORIZED section.

# Validation result:
# STATUS: VALID
# unsupported_items: []

# Result: PASS

In [72]:
%%writefile /content/drive/MyDrive/FleetResolve/data.py

# FleetResolve synthetic operational data

devices = {
    "AX-1001": {
        "device_id": "AX-1001",
        "device_type": "Body Camera",
        "status": "Online",
        "battery": 87,
        "firmware": "5.2.1",
        "last_seen": "2026-08-16 14:20",
        "assigned_user": "USR-001",
    },
    "AX-1002": {
        "device_id": "AX-1002",
        "device_type": "Body Camera",
        "status": "Offline",
        "battery": 8,
        "firmware": "5.1.9",
        "last_seen": "2026-08-16 10:12",
        "assigned_user": "USR-002",
    },
    "AX-1003": {
        "device_id": "AX-1003",
        "device_type": "In-Car Camera",
        "status": "Online",
        "battery": 62,
        "firmware": "5.2.1",
        "last_seen": "2026-08-16 14:25",
        "assigned_user": "USR-003",
    },
    "AX-1004": {
        "device_id": "AX-1004",
        "device_type": "Body Camera",
        "status": "Offline",
        "battery": 76,
        "firmware": "5.0.4",
        "last_seen": "2026-08-16 09:45",
        "assigned_user": "USR-004",
    },
    "AX-1005": {
        "device_id": "AX-1005",
        "device_type": "Sensor",
        "status": "Online",
        "battery": 94,
        "firmware": "3.4.0",
        "last_seen": "2026-08-16 14:28",
        "assigned_user": "USR-005",
    },
}

users = {
    "USR-001": {
        "user_id": "USR-001",
        "name": "Alex Morgan",
        "department": "Patrol",
        "location": "Austin",
    },
    "USR-002": {
        "user_id": "USR-002",
        "name": "Jordan Lee",
        "department": "Patrol",
        "location": "Austin",
    },
    "USR-003": {
        "user_id": "USR-003",
        "name": "Taylor Smith",
        "department": "Traffic",
        "location": "Dallas",
    },
    "USR-004": {
        "user_id": "USR-004",
        "name": "Morgan Chen",
        "department": "Patrol",
        "location": "Dallas",
    },
    "USR-005": {
        "user_id": "USR-005",
        "name": "Casey Brown",
        "department": "Operations",
        "location": "Houston",
    },
}

incidents = {
    "AX-1001": [],
    "AX-1002": [
        {
            "incident_id": "INC-201",
            "type": "Battery Failure",
            "date": "2026-07-20",
            "resolution": "Battery replaced",
        },
        {
            "incident_id": "INC-245",
            "type": "Connectivity Loss",
            "date": "2026-08-02",
            "resolution": "Device restarted",
        },
    ],
    "AX-1003": [],
    "AX-1004": [
        {
            "incident_id": "INC-198",
            "type": "Firmware Issue",
            "date": "2026-07-15",
            "resolution": "Firmware updated",
        },
    ],
    "AX-1005": [],
}

print("FleetResolve data.py created successfully.")

Overwriting /content/drive/MyDrive/FleetResolve/data.py


In [73]:
%%writefile /content/drive/MyDrive/FleetResolve/tools.py

from langchain.tools import tool

from data import devices, users, incidents


@tool
def get_device(device_id: str) -> dict:
    """Get the current status and metadata for a connected device."""
    return devices.get(
        device_id,
        {"error": f"Device {device_id} not found"}
    )


@tool
def get_user(user_id: str) -> dict:
    """Get information about the user assigned to a device."""
    return users.get(
        user_id,
        {"error": f"User {user_id} not found"}
    )


@tool
def get_incident_history(device_id: str) -> list:
    """Get historical incidents associated with a connected device."""
    return incidents.get(device_id, [])


print("FleetResolve tools.py created successfully.")

Overwriting /content/drive/MyDrive/FleetResolve/tools.py


In [74]:
%%writefile /content/drive/MyDrive/FleetResolve/knowledge.py

# FleetResolve operational knowledge

troubleshooting_knowledge = """
Connected Device Troubleshooting Guidelines:

1. A device with battery below 10% should be considered a low-battery condition.

2. A device that is offline with battery below 10% should receive an
   immediate service review recommendation.

3. A previous battery failure indicates that battery health should be
   considered when evaluating a current low-battery condition.

4. A previous connectivity loss indicates that connectivity should be
   considered when evaluating a current offline condition.

5. Recommendations should be based on available evidence. Do not claim
   that a component has failed unless the available evidence supports it.
"""

print("FleetResolve knowledge.py created successfully.")

Overwriting /content/drive/MyDrive/FleetResolve/knowledge.py


In [75]:
%%writefile /content/drive/MyDrive/FleetResolve/policy.py

# FleetResolve decision policy

decision_policy = """
Decision Policy for Connected Devices:

1. If a device has battery below 10%, recommend a service review.

2. If a device is offline AND battery is below 10%, recommend an immediate
   service review.

3. If there is a previous battery failure, recommend checking battery health
   and replacement history.

4. If there is a previous connectivity loss, recommend verifying connectivity.

5. The agent may recommend investigation or service review.

6. The agent must NOT directly authorize component replacement, device
   deactivation, return-to-service, firmware changes, or other consequential
   operational actions.

7. If evidence is insufficient to make a recommendation, the agent must say
   that the evidence is insufficient and request human review.

8. The agent must not introduce specific diagnostic procedures, technical
   causes, or operational steps that are not supported by the available
   evidence or operational knowledge.

9. When the available evidence does not support a specific procedure, the
   agent should state the limitation rather than infer a procedure.
"""

print("FleetResolve policy.py created successfully.")

Overwriting /content/drive/MyDrive/FleetResolve/policy.py


In [76]:
%%writefile /content/drive/MyDrive/FleetResolve/agent.py

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_groq import ChatGroq

from tools import (
    get_device,
    get_user,
    get_incident_history,
)

from knowledge import troubleshooting_knowledge
from policy import decision_policy
from validation import validate_recommendation


# ------------------------------------------------------------
# LLM
# ------------------------------------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


# ------------------------------------------------------------
# Tools available to the agent
# ------------------------------------------------------------

tools = [
    get_device,
    get_user,
    get_incident_history,
]

tool_map = {
    tool.name: tool
    for tool in tools
}

llm_with_tools = llm.bind_tools(tools)


# ------------------------------------------------------------
# Structured FleetResolve investigation
# ------------------------------------------------------------

def investigate_device(device_id: str, max_steps: int = 10):

    structured_evaluation_question = f"""
Investigate device {device_id} and provide an operational recommendation.

Use ONLY the following operational knowledge and decision policy.

OPERATIONAL KNOWLEDGE:
{troubleshooting_knowledge}

DECISION POLICY:
{decision_policy}

Return your answer using exactly these sections:

RECOMMENDATION:
State only the recommendation supported by the evidence and policy.

EVIDENCE:
List only facts retrieved from the available tools.

POLICY BASIS:
Identify the specific guideline or decision-policy rule supporting
the recommendation.

NOT AUTHORIZED:
List consequential actions that require human approval.
Do not present these as recommendations.

IMPORTANT:
- Do not invent policies, thresholds, procedures, technical causes,
  or operational actions.
- Do not use general domain knowledge that is not contained in the
  operational knowledge or decision policy above.
- If the evidence or policy is insufficient, explicitly say so
  and recommend human review.
"""

    messages = [
        HumanMessage(content=structured_evaluation_question)
    ]

    tool_trace = []

    for step in range(max_steps):

        response = llm_with_tools.invoke(messages)

        messages.append(response)

        # Agent has completed the investigation.
        if not response.tool_calls:
            answer = response.content

            recommendation_text = answer

            if "RECOMMENDATION:" in answer and "EVIDENCE:" in answer:
                recommendation_text = answer.split("RECOMMENDATION:", 1)[1]
                recommendation_text = recommendation_text.split("EVIDENCE:", 1)[0]

            validation_result = validate_recommendation(
                recommendation_text
            )

            return {
                "device_id": device_id,
                "answer": response.content,
                "validation": validation_result,
                "tool_trace": tool_trace,
                "steps": step + 1,
            }


        # Execute tools requested by the agent.
        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            selected_tool = tool_map[tool_name]

            result = selected_tool.invoke(tool_args)

            tool_trace.append({
                "tool": tool_name,
                "args": tool_args,
                "result": result,
            })

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

    return {
        "device_id": device_id,
        "answer": (
            "Investigation stopped after reaching the maximum "
            "number of tool-calling steps."
        ),
        "tool_trace": tool_trace,
        "steps": max_steps,
    }


print("FleetResolve structured agent created successfully.")

Overwriting /content/drive/MyDrive/FleetResolve/agent.py


In [77]:
%%writefile /content/drive/MyDrive/FleetResolve/validation.py

def validate_recommendation(recommendation: str) -> dict:
    recommendation_lower = recommendation.lower()

    unsupported_items = []

    for item in [
        "battery replacement",
        "firmware change",
        "device deactivation",
        "return to service",
        "battery under load",
        "signal strength",
        "network settings",
        "charging practices"
    ]:
        if item in recommendation_lower:
            unsupported_items.append(item)

    if unsupported_items:
        return {
            "status": "HUMAN_REVIEW_REQUIRED",
            "reason": "Recommendation contains unsupported operational details.",
            "unsupported_items": unsupported_items
        }

    return {
        "status": "VALID",
        "reason": "Recommendation is within the defined evidence and policy boundary.",
        "unsupported_items": []
    }

print("FleetResolve validation.py created successfully.")

Overwriting /content/drive/MyDrive/FleetResolve/validation.py


In [78]:
%%writefile /content/drive/MyDrive/FleetResolve/backend.py

from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse

from agent import investigate_device

app = FastAPI(
    title="FleetResolve API",
    description="Agentic fleet operations backend",
    version="0.1.0",
)


@app.get("/")
def home():
    return FileResponse(
        "/content/drive/MyDrive/FleetResolve/index.html"
    )


@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "FleetResolve",
    }

@app.get("/devices")
def list_devices():
    from data import devices
    return list(devices.values())

@app.post("/devices/{device_id}/investigate")
def investigate(device_id: str):

    try:
        result = investigate_device(device_id)
        return result

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e),
        )

Overwriting /content/drive/MyDrive/FleetResolve/backend.py


In [79]:
import subprocess
import sys

project_dir = "/content/drive/MyDrive/FleetResolve"

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "backend:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd=project_dir,
)

print("FleetResolve API started.")
print("PID:", process.pid)

FleetResolve API started.
PID: 10884


In [80]:
import subprocess
import sys

project_dir = "/content/drive/MyDrive/FleetResolve"

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "backend:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd=project_dir,
)

print("FleetResolve API starting...")

FleetResolve API starting...


In [81]:
%%writefile /content/drive/MyDrive/FleetResolve/index.html

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>FleetResolve</title>

<style>
    * {
        box-sizing: border-box;
    }

    body {
    margin: 0;
    font-family: Inter, Arial, sans-serif;
    background: #f5f7fa;
    color: #172033;
    height: 100vh;
    overflow: hidden;
    }

    .header {
        background: #172033;
        color: white;
        padding: 18px 28px;
        display: flex;
        justify-content: space-between;
        align-items: center;
    }

    .brand {
        font-size: 22px;
        font-weight: 700;
    }

    .subtitle {
        font-size: 13px;
        opacity: 0.7;
        margin-top: 3px;
    }

    .operator {
        font-size: 14px;
    }

    .container {
    max-width: 1400px;
    margin: 0 auto;
    padding: 20px 28px;
    height: calc(100vh - 70px);
    display: flex;
    flex-direction: column;
    }


    .metrics {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 14px;
    margin-bottom: 16px;
    flex-shrink: 0;
    }

    .metric {
        background: white;
        border: 1px solid #e2e6ec;
        border-radius: 10px;
        padding: 14px 18px;
    }

    .metric-label {
        font-size: 13px;
        color: #667085;
    }

    .metric-value {
        font-size: 28px;
        font-weight: 700;
        margin-top: 8px;
    }

    .main {
    display: grid;
    grid-template-columns: 1.1fr 1.9fr;
    gap: 20px;
    flex: 1;
    min-height: 0;
    }

    .panel {
    background: white;
    border: 1px solid #e2e6ec;
    border-radius: 10px;
    overflow: auto;
    min-height: 0;
    }

    .panel-header {
        padding: 18px 20px;
        border-bottom: 1px solid #e2e6ec;
        font-weight: 700;
    }

    .device {
        padding: 18px 20px;
        border-bottom: 1px solid #edf0f4;
        cursor: pointer;
    }

    .device:hover {
        background: #f8fafc;
    }

    .device.selected {
        background: #eef4ff;
        border-left: 4px solid #356ae6;
    }

    .device-top {
        display: flex;
        justify-content: space-between;
        align-items: center;
    }

    .device-id {
        font-weight: 700;
    }

    .device-type {
        color: #667085;
        font-size: 13px;
        margin-top: 5px;
    }

    .status {
        font-size: 12px;
        font-weight: 600;
        padding: 4px 8px;
        border-radius: 20px;
    }

    .offline {
        background: #fff0f0;
        color: #b42318;
    }

    .online {
        background: #ecfdf3;
        color: #027a48;
    }

    .details {
        padding: 22px;
    }

    .detail-title {
        font-size: 20px;
        font-weight: 700;
    }

    .detail-subtitle {
        color: #667085;
        margin-top: 4px;
    }

    .device-facts {
        display: grid;
        grid-template-columns: repeat(3, 1fr);
        gap: 12px;
        margin: 20px 0;
    }

    .fact {
        background: #f8fafc;
        padding: 12px;
        border-radius: 8px;
    }

    .fact-label {
        font-size: 11px;
        color: #667085;
        text-transform: uppercase;
    }

    .fact-value {
        margin-top: 5px;
        font-weight: 600;
    }

    button {
        border: 0;
        border-radius: 7px;
        padding: 10px 16px;
        font-weight: 600;
        cursor: pointer;
    }

    .primary {
        background: #172033;
        color: white;
    }

    .secondary {
        background: #eef2f6;
        color: #344054;
    }

    button:disabled {
        opacity: 0.55;
        cursor: not-allowed;
    }

    .agent-panel {
        margin-top: 20px;
        border: 1px solid #e2e6ec;
        border-radius: 9px;
        overflow: hidden;
    }

    .agent-header {
        background: #f8fafc;
        padding: 14px 16px;
        font-weight: 700;
        display: flex;
        justify-content: space-between;
    }

    .agent-body {
    padding: 18px 20px;
    font-size: 14px;
    line-height: 1.55;
    text-align: left;
    }

    .agent-section-title {
        margin-top: 18px;
        margin-bottom: 8px;
        font-size: 12px;
        font-weight: 700;
        letter-spacing: 0.08em;
        color: #667085;
    }

    .agent-section-title:first-child {
        margin-top: 0;
    }

    .agent-line {
        margin-bottom: 6px;
    }



    .valid {
        background: #ecfdf3;
        color: #027a48;
    }

    .review {
        background: #fff7ed;
        color: #b54708;
    }

    .trace {
        margin-top: 14px;
        padding: 12px;
        background: #f8fafc;
        border-radius: 7px;
        font-size: 12px;
        color: #475467;
    }

    .human-state {
        margin-top: 20px;
        padding: 16px;
        border: 1px solid #e2e6ec;
        border-radius: 9px;
    }

    .human-state-title {
        font-weight: 700;
        margin-bottom: 12px;
    }

    .state-buttons {
        display: flex;
        gap: 10px;
        flex-wrap: wrap;
    }

    .empty {
        padding: 30px;
        color: #667085;
        text-align: center;
    }

    @media (max-width: 900px) {
        .metrics {
            grid-template-columns: repeat(2, 1fr);
        }

        .main {
            grid-template-columns: 1fr;
        }
    }

    .decision-note {
        margin-top: 10px;
        font-size: 12px;
        color: #98a2b3;
    }

</style>
</head>

<body>

<div class="header">
    <div>
        <div class="brand">FleetResolve</div>
        <div class="subtitle">Agentic fleet operations</div>
    </div>

    <div class="operator">
        Maya . Fleet Operations
    </div>
</div>

<div class="container">

    <div class="metrics">
        <div class="metric">
            <div class="metric-label">Fleet Devices</div>
            <div class="metric-value">5</div>
        </div>

        <div class="metric">
            <div class="metric-label">Online</div>
            <div class="metric-value">3</div>
        </div>

        <div class="metric">
            <div class="metric-label">Needs Investigation</div>
            <div class="metric-value">2</div>
        </div>

        <div class="metric">
            <div class="metric-label">Agent Investigations</div>
            <div class="metric-value" id="investigationCount">0</div>
        </div>
    </div>

    <div class="main">

        <div class="panel">
            <div class="panel-header">
                Connected Devices
            </div>

            <div id="deviceList"></div>
        </div>

        <div class="panel">

            <div id="details" class="empty">
                Select a device to investigate.
            </div>

        </div>

    </div>

</div>

<script>

async function loadDevices() {
    const deviceList = document.getElementById("deviceList");

    try {
        const response = await fetch("/devices");

        if (!response.ok) {
            throw new Error(
                `Backend returned HTTP ${response.status}`
            );
        }

        const devices = await response.json();

        deviceList.innerHTML = "";

        devices.forEach(device => {

            const deviceCard = document.createElement("div");

            deviceCard.className = "device-card";

            deviceCard.innerHTML = `
                <div class="device-id">
                    ${device.device_id}
                </div>

                <div class="device-type">
                    ${device.device_type}
                </div>

                <div class="device-status">
                    ${device.status}
                    · ${device.battery}% battery
                </div>
            `;

            deviceCard.onclick = () => {
                investigateDevice(device.device_id);
            };

            deviceList.appendChild(deviceCard);
        });

    } catch (error) {

        deviceList.innerHTML = `
            <div class="error">
                Unable to load devices.
            </div>
        `;

        console.error(error);
    }
}

loadDevices();

/*const devices = [
    {
        id: "AX-1001",
        type: "Body Camera",
        status: "Online",
        battery: 87,
        firmware: "5.2.1"
    },
    {
        id: "AX-1002",
        type: "Body Camera",
        status: "Offline",
        battery: 8,
        firmware: "5.1.9"
    },
    {
        id: "AX-1003",
        type: "In-Car Camera",
        status: "Online",
        battery: 62,
        firmware: "5.2.1"
    },
    {
        id: "AX-1004",
        type: "Body Camera",
        status: "Offline",
        battery: 76,
        firmware: "5.0.4"
    },
    {
        id: "AX-1005",
        type: "Sensor",
        status: "Online",
        battery: 94,
        firmware: "3.4.0"
    }
];
*/

let devices =[];
let selectedDevice = null;
let investigationCount = 0;

async function renderDevices() {

    const list = document.getElementById("deviceList");

    try {

        const response = await fetch("/devices");

        if (!response.ok) {
            throw new Error(
                `Backend returned HTTP ${response.status}`
            );
        }

        devices = await response.json();

        // Put devices needing investigation first.
        devices.sort((a, b) => {
            if (a.status === "Offline" && b.status !== "Offline") return -1;
            if (a.status !== "Offline" && b.status === "Offline") return 1;
            return 0;
        });

        list.innerHTML = "";

        devices.forEach(device => {

            const deviceElement = document.createElement("div");

            deviceElement.className = "device";

            const isOffline = device.status === "Offline";

            deviceElement.innerHTML = `
                <div class="device-top">

                    <div>
                        <div class="device-id">
                            ${device.device_id}
                        </div>

                        <div class="device-type">
                            ${device.device_type}
                        </div>
                    </div>

                    <div class="status ${isOffline ? "offline" : "online"}">
                        ${device.status}
                    </div>

                </div>

                <div style="margin-top: 8px; font-size: 13px;">
                    ${device.battery}% battery
                </div>

                ${
                    isOffline
                        ? `
                            <div style="
                                margin-top: 8px;
                                font-size: 12px;
                                font-weight: 600;
                                color: #b42318;
                            ">
                                ⚠ Needs investigation
                            </div>
                          `
                        : `
                            <div style="
                                margin-top: 8px;
                                font-size: 12px;
                                color: #667085;
                            ">
                                Investigate available
                            </div>
                          `
                }
            `;

            deviceElement.addEventListener(
                "click",
                () => selectDevice(device.device_id)
            );

            list.appendChild(deviceElement);
        });

    } catch (error) {

        console.error(
            "Unable to load devices:",
            error
        );

        list.innerHTML = `
            <div class="empty">
                Unable to load devices.
            </div>
        `;
    }
}

function selectDevice(deviceId) {

    selectedDevice = deviceId;
    const details = document.getElementById("details");

    details.classList.remove("empty");

    const device = devices.find(d => d.device_id === deviceId);


    details.innerHTML = `

        <div class="details">

            <div class="detail-title">
                ${device.device_id}
            </div>

            <div class="detail-subtitle">
                ${device.device_type}
            </div>

            <div class="device-facts">

                <div class="fact">
                    <div class="fact-label">Status</div>
                    <div class="fact-value">${device.status}</div>
                </div>

                <div class="fact">
                    <div class="fact-label">Battery</div>
                    <div class="fact-value">${device.battery}%</div>
                </div>

                <div class="fact">
                    <div class="fact-label">Firmware</div>
                    <div class="fact-value">${device.firmware}</div>
                </div>

            </div>

            <button
                class="primary"
                onclick="investigateDevice('${device.device_id}')">
                Investigate with Agent
            </button>

            <div id="agentResult"></div>

            <div class="human-state">

                <div class="human-state-title">
                    Maya's Decision
                </div>

                <div class="state-buttons">

                    <button
                        class="secondary"
                        disabled>
                        Investigated by Maya
                    </button>

                    <button
                        class="secondary"
                        disabled>
                        Resolved
                    </button>

                    <button
                        class="secondary"
                        disabled>
                        Verified
                    </button>

                </div>

                <div class="decision-note">
                     Human decision workflow not enabled in prototype.
                </div>

                <div id="humanState"
                     style="margin-top:12px;color:#667085;">
                    Awaiting human action
                </div>

            </div>

        </div>
    `;

    renderDevices();
}

async function investigateDevice(deviceId) {

    const resultContainer =
        document.getElementById("agentResult");

    resultContainer.innerHTML = `
        <div class="agent-panel">
            <div class="agent-header">
                <span>FleetResolve Agent</span>
                <span>Investigating...</span>
            </div>

            <div class="agent-body">
                Gathering device evidence...
            </div>
        </div>
    `;

    try {

        /*
         * DEMO MODE
         *
         * The live API has already been validated in Colab.
         * Demo mode lets the LinkedIn prototype run without
         * requiring a public tunnel to the Colab backend.
         */

        /*const demoResult = {

            device_id: "AX-1002",

            answer: `
RECOMMENDATION:
Initiate an immediate service review of device AX-1002.
As part of the review, verify the device's battery health
and replacement history and confirm its connectivity status.

EVIDENCE:
- Device status: Offline.
- Battery level: 8% (below 10%).
- Previous Battery Failure: INC-201.
- Previous Connectivity Loss: INC-245.

POLICY BASIS:
- Offline AND battery below 10% → immediate service review.
- Previous battery failure → check battery health and replacement history.
- Previous connectivity loss → verify connectivity.

NOT AUTHORIZED:
- Battery replacement.
- Device deactivation.
- Firmware changes or updates.
- Return-to-service approval.
- Other consequential operational actions requiring human approval.
            `,

            validation: {
                status: "VALID",
                reason:
                    "Recommendation is within the defined evidence and policy boundary.",
                unsupported_items: []
            },

            tool_trace: [
                { tool: "get_device" },
                { tool: "get_incident_history" }
            ]
        };
        */



         const response = await fetch(
            `/devices/${deviceId}/investigate`,
            { method: "POST" }
          );
         if (!response.ok) {
            throw new Error(
        `   Backend returned HTTP ${response.status}`
          );
          }
          const result = await response.json();


        investigationCount++;

        document.getElementById(
            "investigationCount"
        ).textContent = investigationCount;

        const validation = result.validation;

        const validationClass =
            validation.status === "VALID"
                ? "valid"
                : "review";

        const validationLabel =
            validation.status === "VALID"
                ? "✓ Recommendation validated"
                : "⚠ Human review required";

        resultContainer.innerHTML = `

            <div class="agent-panel">

                <div class="agent-header">
                    <span>FleetResolve Agent</span>
                    <span>${validationLabel}</span>
                </div>

                <div class="validation ${validationClass}">
                    <strong>${validation.status}</strong><br>
                    ${validation.reason}
                </div>

                <div class="agent-body">
                    ${renderAgentAnswer(result.answer)}
                </div>

                <div class="trace">
                    <strong>Agent tool trace</strong><br><br>
                    ${result.tool_trace
                        .map(item => `• ${item.tool}`)
                        .join("<br>")}
                </div>

            </div>
        `;

    } catch (error) {

        resultContainer.innerHTML = `
            <div class="agent-panel">
                <div class="agent-header">
                    FleetResolve Agent
                </div>

                <div class="agent-body">
                    Unable to run the investigation.
                </div>
            </div>
        `;
    }
}

function setHumanState(state) {

    document.getElementById("humanState").innerHTML =
        `<strong>${state}</strong>`;
}



function renderAgentAnswer(answer) {

    return answer
        .replace(/\*\*/g, "")
        .split("\n")
        .map(line => line.trim())
        .filter(line => line.length > 0)
        .map(line => {

            const sectionTitles = [
                "RECOMMENDATION:",
                "EVIDENCE:",
                "POLICY BASIS:",
                "NOT AUTHORIZED:"
            ];

            if (sectionTitles.includes(line)) {
                return `
                    <div class="agent-section-title">
                        ${escapeHtml(line.replace(":", ""))}
                    </div>
                `;
            }

            return `
                <div class="agent-line">
                    ${escapeHtml(line.replace(/^[-•]\s*/, ""))}
                </div>
            `;

        })
        .join("");
}


function escapeHtml(text) {

    const div = document.createElement("div");

    div.textContent = text;

    return div.innerHTML;
}

renderDevices();

</script>

</body>
</html>

Overwriting /content/drive/MyDrive/FleetResolve/index.html


In [82]:
import subprocess
import sys

project_dir = "/content/drive/MyDrive/FleetResolve"

web_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "http.server",
        "8080",
    ],
    cwd=project_dir,
)

print("FleetResolve UI server started.")
print("Process ID:", web_process.pid)
print("Serving index.html on port 8080")

FleetResolve UI server started.
Process ID: 10886
Serving index.html on port 8080


In [83]:
from google.colab.output import eval_js

ui_url = eval_js("google.colab.kernel.proxyPort(8000)")

print("FleetResolve UI:")
print(ui_url)

FleetResolve UI:
https://8000-m-s-kkb-use1b1-2oh3t86i3l3hd-b.us-east1-1.prod.colab.dev


In [84]:
import subprocess
import sys

web_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "http.server",
        "8080",
        "--directory",
        "/content/drive/MyDrive/FleetResolve",
    ]
)

print("UI server restarted.")

UI server restarted.
